In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


ruta = Path("data/REHAB/Rehab_exercise/d02_processed_data")

# ------------------------------------------------------------
# Canales según el artículo
# ------------------------------------------------------------

canales_1 = [
    "pitch1",
    "yaw1",
    "roll1",
    "pitch2",
    "yaw2",
    "roll2"
]

canales_2 = [
    "f1",
    "f2",
    "f3",
    "f4",
    "f5",
    "pitch3"
]

# Se eliminaron: mean, range, rms y energy
estadisticas = [
    "std",
    "median",
    "min",
    "max",
    "iqr",
    "mad_diff"
]

NUM_VENTANAS = 8


# ------------------------------------------------------------
# Función para extraer features de una señal 1D
# ------------------------------------------------------------

def extraer_features(x):
    """
    Extrae seis características estadísticas de una señal 1D.
    """

    x = np.asarray(x)

    q25 = np.percentile(x, 25)
    q75 = np.percentile(x, 75)

    diffs = np.diff(x)

    return {
        "std": np.std(x),
        "median": np.median(x),
        "min": np.min(x),
        "max": np.max(x),
        "iqr": q75 - q25,
        "mad_diff": np.mean(np.abs(diffs))
    }


# ------------------------------------------------------------
# Construcción del DataFrame
# ------------------------------------------------------------

filas = []

# Movimientos 000 a 015, ignorando 014
movimientos = [
    f"{i:03d}"
    for i in range(16)
    if i != 14
]

for movimiento in movimientos:

    archivo_1 = ruta / f"{movimiento}_1.npy"
    archivo_2 = ruta / f"{movimiento}_2.npy"

    # --------------------------------------------------------
    # Validar existencia
    # --------------------------------------------------------

    if not archivo_1.exists():
        print(f"No existe: {archivo_1.name}")
        continue

    if not archivo_2.exists():
        print(f"No existe: {archivo_2.name}")
        continue

    # --------------------------------------------------------
    # Cargar ambos archivos
    # --------------------------------------------------------

    try:
        datos_1 = np.load(archivo_1)
        datos_2 = np.load(archivo_2)

    except Exception as e:
        print(f"Error en movimiento {movimiento}: {e}")
        continue

    # Esperamos:
    # datos_1 -> (n_repeticiones, 880, 6)
    # datos_2 -> (n_repeticiones, 880, 6)

    if datos_1.ndim != 3 or datos_2.ndim != 3:
        print(
            f"{movimiento}: dimensiones inesperadas "
            f"{datos_1.shape}, {datos_2.shape}"
        )
        continue

    # --------------------------------------------------------
    # Verificar correspondencia entre ambos archivos
    # --------------------------------------------------------

    if datos_1.shape[0] != datos_2.shape[0]:
        print(
            f"ERROR {movimiento}: diferente número de repeticiones "
            f"_1={datos_1.shape[0]}, _2={datos_2.shape[0]}"
        )
        continue

    if datos_1.shape[1] != datos_2.shape[1]:
        print(
            f"ERROR {movimiento}: diferente número de tiempos "
            f"_1={datos_1.shape[1]}, _2={datos_2.shape[1]}"
        )
        continue

    if datos_1.shape[2] != 6 or datos_2.shape[2] != 6:
        print(
            f"ERROR {movimiento}: se esperaban 6 canales por archivo"
        )
        continue

    n_repeticiones = datos_1.shape[0]
    n_tiempos = datos_1.shape[1]

    # En este dataset: 880 / 8 = 110
    if n_tiempos % NUM_VENTANAS != 0:
        print(
            f"ERROR {movimiento}: {n_tiempos} tiempos no pueden "
            f"dividirse exactamente en {NUM_VENTANAS} ventanas"
        )
        continue

    tamano_ventana = n_tiempos // NUM_VENTANAS

    print(
        f"{movimiento}: {n_repeticiones} repeticiones "
        f"| {NUM_VENTANAS} ventanas de {tamano_ventana} tiempos "
        f"| filas generadas: {n_repeticiones * NUM_VENTANAS}"
    )

    # --------------------------------------------------------
    # Recorrer repeticiones
    # --------------------------------------------------------

    for rep in range(n_repeticiones):

        muestra_1 = datos_1[rep]  # (880, 6)
        muestra_2 = datos_2[rep]  # (880, 6)

        # Identificador único dentro de todo el dataset
        repeticion_id = f"{movimiento}_{rep + 1:04d}"

        # ----------------------------------------------------
        # Dividir cada repetición en 8 ventanas
        # ----------------------------------------------------

        for numero_ventana in range(NUM_VENTANAS):

            inicio = numero_ventana * tamano_ventana
            fin = inicio + tamano_ventana

            # Ambas ventanas representan el mismo intervalo
            ventana_1 = muestra_1[inicio:fin, :]  # (110, 6)
            ventana_2 = muestra_2[inicio:fin, :]  # (110, 6)

            fila = {
                "movimiento": movimiento,
                "repeticion_id": repeticion_id,
                "ventana": numero_ventana + 1
            }

            # ------------------------------------------------
            # Features de *_1.npy
            # ------------------------------------------------

            for i, canal in enumerate(canales_1):

                x = ventana_1[:, i]
                features = extraer_features(x)

                for nombre_stat, valor in features.items():
                    fila[f"{canal}_{nombre_stat}"] = valor

            # ------------------------------------------------
            # Features de *_2.npy
            # ------------------------------------------------

            for i, canal in enumerate(canales_2):

                x = ventana_2[:, i]
                features = extraer_features(x)

                for nombre_stat, valor in features.items():
                    fila[f"{canal}_{nombre_stat}"] = valor

            filas.append(fila)


# ------------------------------------------------------------
# DataFrame final
# ------------------------------------------------------------

df_final = pd.DataFrame(filas)

if not df_final.empty:

    df_final["movimiento"] = df_final["movimiento"].astype("category")
    df_final["repeticion_id"] = df_final["repeticion_id"].astype("category")
    df_final["ventana"] = df_final["ventana"].astype("int8")

    # Ordenar las filas para facilitar su lectura
    df_final = (
        df_final
        .sort_values(["movimiento", "repeticion_id", "ventana"])
        .reset_index(drop=True)
    )

print("\nDimensiones finales:", df_final.shape)
display(df_final.head(16))

000: 232 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 1856
001: 212 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 1696
002: 267 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2136
003: 250 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2000
004: 287 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2296
005: 293 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2344
006: 260 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2080
007: 385 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 3080
008: 299 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2392
009: 307 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2456
010: 311 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2488
011: 235 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 1880
012: 293 repeticiones | 8 ventanas de 110 tiempos | filas generadas: 2344
013: 313 repeticiones | 8 ventanas de 

,movimiento,repeticion_id,ventana,pitch1_std,pitch1_median,pitch1_min,pitch1_max,pitch1_iqr,pitch1_mad_diff,yaw1_std,...,f5_min,f5_max,f5_iqr,f5_mad_diff,pitch3_std,pitch3_median,pitch3_min,pitch3_max,pitch3_iqr,pitch3_mad_diff
0,000,000_0001,1,27.407620,-14.465941,-49.403141,27.532859,54.811625,1.022145,6.225978,...,-9.145568,6.354432,9.825,0.259633,69.144078,43.791327,-95.283864,85.718660,135.985946,2.531138
1,000,000_0001,2,32.124926,28.154059,-42.124041,55.319659,58.130575,1.142213,6.924127,...,-5.945568,5.254432,1.000,0.141284,70.721517,-78.907298,-91.496040,102.666751,91.967748,2.549152
2,000,000_0001,3,19.543044,-31.069241,-48.141441,13.094959,37.844550,0.612022,5.851873,...,-6.945568,4.254432,3.275,0.311009,68.491422,79.377449,-81.726537,112.044924,113.431176,3.701181
3,000,000_0001,4,16.822138,29.793209,-18.259741,50.364559,18.295750,0.958684,7.014800,...,-2.745568,0.054432,1.450,0.092661,75.773251,-51.502727,-82.993920,111.134494,156.478068,2.085267
4,000,000_0001,5,15.094130,-39.042191,-51.259241,4.181559,24.137800,0.798237,8.785314,...,-6.545568,4.254432,4.275,0.211009,80.641894,-73.154802,-83.343424,113.818229,166.163633,2.149617
5,000,000_0001,6,10.770647,32.556259,5.981159,49.715159,16.010200,0.882846,8.933424,...,-6.345568,2.454432,2.900,0.218349,67.791173,75.060499,-79.462208,117.690851,107.967021,3.744332
6,000,000_0001,7,16.710919,-39.794541,-51.937941,14.465359,25.009025,0.956092,10.113418,...,-6.745568,0.254432,2.775,0.116514,67.268070,17.654425,-79.866143,88.519277,134.236563,2.104532
7,000,000_0001,8,14.742844,33.242009,-13.011541,47.779459,17.444200,1.011986,9.507459,...,-9.245568,4.154432,1.600,0.238532,71.890751,-66.551177,-82.865577,103.640207,143.421793,2.408257
8,000,000_0002,1,16.524316,-34.320084,-45.946884,2.275216,30.938250,0.797305,6.954146,...,-9.437955,5.562045,3.950,0.244954,66.780978,-78.459407,-93.939868,88.654906,84.955460,2.250026
9,000,000_0002,2,13.662016,31.987766,3.031316,52.559116,18.295750,0.895542,7.529208,...,-6.637955,3.762045,4.475,0.246789,87.008565,-35.171014,-85.437747,109.601097,181.526208,2.116790


In [3]:
df_final.to_csv("data/df_final.csv", index=False)